In [ ]:
import langchain
print("LangChain version:", langchain.__version__)

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# os.environ['GEMINI_API_KEY']=os.getenv("GEMINI_API_KEY")
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
# print("GEMINI_API_KEY:", os.environ['GEMINI_API_KEY'])
print("GROQ_API_KEY:", os.environ['GROQ_API_KEY']) 

In [ ]:

from langchain.chat_models import init_chat_model
model = init_chat_model(model="llama-3.1-8b-instant", model_provider="groq",
                            api_key=os.getenv("GROQ_API_KEY"))

In [ ]:
response = model.invoke("Hello, how are you in one word?")

In [ ]:
response

In [ ]:
from langchain_groq import ChatGroq
chat_groq = ChatGroq(model="llama-3.1-8b-instant", api_key=os.getenv("GROQ_API_KEY"))
response_groq = chat_groq.invoke("Hello, how are you in one word?")

In [ ]:
response

In [ ]:
response_groq

In [ ]:
import json

def summarize_resp(obj):
    """Extract common non-content fields from LangChain-style response objects or dicts."""
    meta = {}
    # response_metadata may be an attribute or a dict key
    rm = None
    try:
        rm = getattr(obj, 'response_metadata', None) or (obj.get('response_metadata') if isinstance(obj, dict) else None)
    except Exception:
        rm = None

    meta['id'] = getattr(obj, 'id', None) or (obj.get('id') if isinstance(obj, dict) else None)

    if isinstance(rm, dict):
        meta['model_name'] = rm.get('model_name')
        meta['model_provider'] = rm.get('model_provider')
        meta['service_tier'] = rm.get('service_tier')
        meta['system_fingerprint'] = rm.get('system_fingerprint')
        meta['finish_reason'] = rm.get('finish_reason')
        tu = rm.get('token_usage') or {}
        meta['prompt_tokens'] = tu.get('prompt_tokens')
        meta['completion_tokens'] = tu.get('completion_tokens')
        meta['total_tokens'] = tu.get('total_tokens')
        meta['completion_time'] = tu.get('completion_time')
        meta['total_time'] = tu.get('total_time')
        meta['queue_time'] = tu.get('queue_time')
    else:
        # try to find similar attributes directly on object
        meta['finish_reason'] = getattr(obj, 'finish_reason', None)
        # best-effort token attributes
        meta['total_tokens'] = getattr(obj, 'total_tokens', None)
        meta['completion_time'] = getattr(obj, 'completion_time', None)
        meta['total_time'] = getattr(obj, 'total_time', None)

    # fallback: usage_metadata
    um = getattr(obj, 'usage_metadata', None) or (obj.get('usage_metadata') if isinstance(obj, dict) else None)
    if isinstance(um, dict):
        meta['usage_input_tokens'] = um.get('input_tokens')
        meta['usage_output_tokens'] = um.get('output_tokens')
        meta['usage_total_tokens'] = um.get('total_tokens')

    return meta

def pretty_print(name, obj):
    print('---', name, '---')
    print(json.dumps(summarize_resp(obj), indent=2, default=str))
    print()

def score_meta(m):
    """Score a summarized meta dict. Higher is better. Adjust weights to prefer cost/latency/reliability."""
    s = 0.0
    # Reliability: prefer finished runs
    if m.get('finish_reason') == 'stop':
        s += 50.0
    elif m.get('finish_reason') is not None:
        s -= 20.0

    # Cost: penalize total tokens (if available)
    total_tokens = m.get('total_tokens') or m.get('usage_total_tokens') or 0
    s -= (total_tokens) * 0.5  # token cost weight (tune as needed)

    # Latency: penalize total_time, completion_time, and queue_time
    total_time = m.get('total_time') or 0
    s -= total_time * 10.0
    completion_time = m.get('completion_time') or 0
    s -= completion_time * 8.0
    queue_time = m.get('queue_time') or 0
    s -= queue_time * 5.0

    # small bonus if model_name present
    if m.get('model_name'):
        s += 2.0

    return s

# Collect metas for available responses
metas = {}
for name in ('response_groq', 'response', 'resp'):
    if name in globals():
        metas[name] = summarize_resp(globals()[name])

if not metas:
    print('No responses to compare. Run the model cells first and re-run this cell.')
else:
    scores = {n: score_meta(m) for n, m in metas.items()}
    # print table
    for n, m in metas.items():
        print(f"{n}: score={scores[n]:.2f}")
        print(json.dumps(m, indent=2, default=str))
        print()
    winner = max(scores, key=scores.get)
    print('--- Recommendation ---')
    print('Best response by current scoring:', winner)
    print('Score:', scores[winner])
    print('Winner details:')
    print(json.dumps(metas[winner], indent=2, default=str))

    print('\nTo change the decision logic, edit the score_meta() function weights (token cost / latency / finish_reason).')